In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


### Load DataSet

In [5]:
df_raw = pd.read_csv('results.csv')
print("Shape of the dataset:", df_raw.shape)

Shape of the dataset: (49477, 9)


In [6]:
print(df_raw.dtypes)
print()

date              str
home_team         str
away_team         str
home_score    float64
away_score    float64
tournament        str
city              str
country           str
neutral          bool
dtype: object



### Data Processing

In [7]:
print("Missing values in the dataset:")
print(df_raw.isnull().sum())

Missing values in the dataset:
date           0
home_team      0
away_team      0
home_score    44
away_score    44
tournament     0
city           0
country        0
neutral        0
dtype: int64


In [ ]:
# Try the naive approach first, to see exactly what goes wrong
try:
    pd.to_datetime(df_raw['date'])
    print("Parsed fine, no cleaning needed.")
except Exception as e:
    print("FAILED with naive parsing:")
    print(e)

FAILED with naive parsing:
time data "03-02-00" doesn't match format "%Y-%m-%d". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.


In [10]:
# Diagnose: how many rows are in ISO format (YYYY-MM-DD) vs the other format?
import re
iso_pattern = re.compile(r'^\d{4}-\d{2}-\d{2}$')
is_iso = df_raw['date'].astype(str).str.match(iso_pattern)

print("Rows in YYYY-MM-DD format:", is_iso.sum())
print("Rows in the other format:", (~is_iso).sum())
print()
print("Example ISO-format date:", df_raw.loc[is_iso, 'date'].iloc[0])
print("Example non-ISO date:", df_raw.loc[~is_iso, 'date'].iloc[0])

Rows in YYYY-MM-DD format: 127
Rows in the other format: 49350

Example ISO-format date: 1872-11-30
Example non-ISO date: 03-02-00


In [11]:
def clean_dates(df):
    """
    Parses the mixed-format date column correctly:
    - YYYY-MM-DD rows are parsed directly
    - DD-MM-YY rows are parsed with explicit century assignment, using the single
      real rollover point in the data (99 -> 00 around the year 1999/2000) rather
      than trusting pandas' generic 2-digit year guess (which would misdate ~7,400 rows).
    """
    df = df.copy()
    iso_pattern = re.compile(r'^\d{4}-\d{2}-\d{2}$')
    is_iso = df['date'].astype(str).str.match(iso_pattern)

    df['date_clean'] = pd.NaT
    df.loc[is_iso, 'date_clean'] = pd.to_datetime(df.loc[is_iso, 'date'], format='%Y-%m-%d')

    non_iso_df = df.loc[~is_iso].copy()
    split = non_iso_df['date'].str.split('-', expand=True)
    non_iso_df['day'] = split[0].astype(int)
    non_iso_df['month'] = split[1].astype(int)
    non_iso_df['yy'] = split[2].astype(int)

    yy_vals = non_iso_df['yy'].tolist()
    rollover = next(
        i for i in range(1, len(yy_vals))
        if yy_vals[i - 1] >= 90 and yy_vals[i] <= 10 and yy_vals[i] < yy_vals[i - 1]
    )
    non_iso_df['century'] = [1900 if pos < rollover else 2000 for pos in range(len(non_iso_df))]
    non_iso_df['year'] = non_iso_df['century'] + non_iso_df['yy']
    non_iso_df['date_clean'] = pd.to_datetime(non_iso_df[['year', 'month', 'day']])

    df.loc[~is_iso, 'date_clean'] = non_iso_df['date_clean'].values
    df['date_clean'] = pd.to_datetime(df['date_clean'])
    return df

df_clean = clean_dates(df_raw)
print("Cleaned date range:", df_clean['date_clean'].min(), "to", df_clean['date_clean'].max())
print("Correctly sortable:", df_clean.sort_values('date_clean')['date_clean'].is_monotonic_increasing)

Cleaned date range: 1872-11-30 00:00:00 to 2026-06-27 00:00:00
Correctly sortable: True


In [12]:
# The 44 rows with missing scores are the actual upcoming 2026 World Cup fixtures,
# not bad data. Set them aside as a genuine prediction set instead of dropping them.
upcoming_fixtures = df_clean[df_clean['home_score'].isnull()].copy()
df_clean = df_clean.dropna(subset=['home_score', 'away_score']).copy()
print(f"Training data: {len(df_clean)} rows")
print(f"Upcoming fixtures set aside: {len(upcoming_fixtures)} rows")

Training data: 49433 rows
Upcoming fixtures set aside: 44 rows


In [13]:
# Save the cleaned file
df_clean_export = df_clean.drop(columns=['date']).rename(columns={'date_clean': 'date'})
df_clean_export = df_clean_export.sort_values('date').reset_index(drop=True)
df_clean_export.to_csv('results_cleaned.csv', index=False)
print(f"Saved results_cleaned.csv with {len(df_clean_export)} rows")

Saved results_cleaned.csv with 49433 rows


### load the cleaned file and define target

In [15]:
df = pd.read_csv('results_cleaned.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

def get_outcome(row):
    if row['home_score'] > row['away_score']:
        return 'Home Win'
    elif row['home_score'] < row['away_score']:
        return 'Away Win'
    else:
        return 'Draw'

df['match_outcome'] = df.apply(get_outcome, axis=1)

historical_df = df[df['date'].dt.year >= 1998].copy().reset_index(drop=True)
print(f"[+] {len(historical_df)} matches from 1998 onward, correctly sorted.")
print(historical_df['match_outcome'].value_counts(normalize=True).round(3))

[+] 26908 matches from 1998 onward, correctly sorted.
match_outcome
Home Win    0.482
Away Win    0.284
Draw        0.234
Name: proportion, dtype: float64


## Feature Engineering

In [19]:
df_features = historical_df.copy()

# Trackers
team_stats = {}      # rolling form: (date, goals_for, goals_against, won)
h2h_stats = {}       # head-to-head: (date, home_won, home_team_name)
elo_ratings = {}      # current Elo rating per team

INITIAL_ELO = 1500   # starting rating for any team with no history
K = 20               # how fast Elo reacts to new results

home_win_rate, away_win_rate = [], []
home_avg_goals_for, away_avg_goals_for = [], []
home_avg_goals_against, away_avg_goals_against = [], []
h2h_home_win_rate = []
days_since_last_home, days_since_last_away = [], []
home_elo_before, away_elo_before = [], []

print("[*] Calculating rolling team statistics and Elo ratings... This may take a moment.")

for idx, row in df_features.iterrows():
    h_team, a_team, match_date = row['home_team'], row['away_team'], row['date']

    for team in [h_team, a_team]:
        if team not in team_stats:
            team_stats[team] = []

    pair_key = tuple(sorted([h_team, a_team]))
    if pair_key not in h2h_stats:
        h2h_stats[pair_key] = []

    # --- Rolling form (last 10 matches) ---
    h_hist = [m for m in team_stats[h_team] if m[0] < match_date][-10:]
    if h_hist:
        home_win_rate.append(np.mean([m[3] for m in h_hist]))
        home_avg_goals_for.append(np.mean([m[1] for m in h_hist]))
        home_avg_goals_against.append(np.mean([m[2] for m in h_hist]))
        days_since_last_home.append((match_date - h_hist[-1][0]).days)
    else:
        home_win_rate.append(0.33)
        home_avg_goals_for.append(1.0)
        home_avg_goals_against.append(1.0)
        days_since_last_home.append(365)

    a_hist = [m for m in team_stats[a_team] if m[0] < match_date][-10:]
    if a_hist:
        away_win_rate.append(np.mean([m[3] for m in a_hist]))
        away_avg_goals_for.append(np.mean([m[1] for m in a_hist]))
        away_avg_goals_against.append(np.mean([m[2] for m in a_hist]))
        days_since_last_away.append((match_date - a_hist[-1][0]).days)
    else:
        away_win_rate.append(0.33)
        away_avg_goals_for.append(1.0)
        away_avg_goals_against.append(1.0)
        days_since_last_away.append(365)

    # --- Head-to-head ---
    h2h_hist = [m for m in h2h_stats[pair_key] if m[0] < match_date]
    if h2h_hist:
        h2h_home_results = [m[1] for m in h2h_hist if m[2] == h_team]
        h2h_home_win_rate.append(np.mean(h2h_home_results) if h2h_home_results else 0.5)
    else:
        h2h_home_win_rate.append(0.5)

    # --- Elo (rating BEFORE this match, to avoid leaking the result into the feature) ---
    h_elo = elo_ratings.get(h_team, INITIAL_ELO)
    a_elo = elo_ratings.get(a_team, INITIAL_ELO)
    home_elo_before.append(h_elo)
    away_elo_before.append(a_elo)

    expected_h = 1 / (1 + 10 ** ((a_elo - h_elo) / 400))

    # --- Actual result, shared by team_stats, h2h_stats, and Elo update ---
    h_won = 1 if row['home_score'] > row['away_score'] else 0
    a_won = 1 if row['away_score'] > row['home_score'] else 0
    score_h = 1.0 if h_won else (0.0 if a_won else 0.5)

    # Update Elo ratings based on gap between actual and expected result
    elo_ratings[h_team] = h_elo + K * (score_h - expected_h)
    elo_ratings[a_team] = a_elo + K * ((1 - score_h) - (1 - expected_h))

    # Log this match into the trackers for future loop iterations
    team_stats[h_team].append((match_date, row['home_score'], row['away_score'], h_won))
    team_stats[a_team].append((match_date, row['away_score'], row['home_score'], a_won))
    h2h_stats[pair_key].append((match_date, h_won, h_team))

# Attach engineered arrays back to the feature DataFrame
df_features['home_rolling_win_rate'] = home_win_rate
df_features['away_rolling_win_rate'] = away_win_rate
df_features['home_avg_goals_for'] = home_avg_goals_for
df_features['away_avg_goals_for'] = away_avg_goals_for
df_features['home_avg_goals_against'] = home_avg_goals_against
df_features['away_avg_goals_against'] = away_avg_goals_against
df_features['h2h_home_win_rate'] = h2h_home_win_rate
df_features['days_since_last_home'] = days_since_last_home
df_features['days_since_last_away'] = days_since_last_away
df_features['home_elo'] = home_elo_before
df_features['away_elo'] = away_elo_before

print("[+] Rolling statistics and Elo ratings complete.")
print()
print("Current top 10 team Elo ratings:")
print(pd.Series(elo_ratings).sort_values(ascending=False).head(10).round(1))

[*] Calculating rolling team statistics and Elo ratings... This may take a moment.
[+] Rolling statistics and Elo ratings complete.

Current top 10 team Elo ratings:
Argentina    1958.1
Spain        1933.0
France       1905.1
Brazil       1882.9
England      1868.8
Portugal     1857.0
Colombia     1856.6
Germany      1850.1
Japan        1845.2
Morocco      1843.6
dtype: float64


In [20]:
# Differential features (the gap between the two competing sides)
df_features['win_rate_diff'] = df_features['home_rolling_win_rate'] - df_features['away_rolling_win_rate']
df_features['attack_diff'] = df_features['home_avg_goals_for'] - df_features['away_avg_goals_for']
df_features['defense_diff'] = df_features['home_avg_goals_against'] - df_features['away_avg_goals_against']
df_features['rest_diff'] = df_features['days_since_last_home'] - df_features['days_since_last_away']
df_features['elo_diff'] = df_features['home_elo'] - df_features['away_elo']
df_features['is_world_cup'] = (df_features['tournament'] == 'FIFA World Cup').astype(int)

print("[+] Feature engineering complete!")
display(df_features[['date', 'home_team', 'away_team', 'elo_diff', 'win_rate_diff',
                      'defense_diff', 'h2h_home_win_rate', 'match_outcome']].tail())

[+] Feature engineering complete!


,date,home_team,away_team,elo_diff,win_rate_diff,defense_diff,h2h_home_win_rate,match_outcome
26903,2026-06-17,Ghana,Panama,-116.851882,-0.1,-0.2,0.5,Home Win
26904,2026-06-18,Switzerland,Bosnia and Herzegovina,190.289305,0.2,-0.2,0.0,Home Win
26905,2026-06-18,Czech Republic,South Africa,49.070976,0.0,-0.1,0.5,Draw
26906,2026-06-18,Mexico,South Korea,15.459756,0.1,-0.9,0.4,Home Win
26907,2026-06-18,Canada,Qatar,141.627972,0.2,-0.9,1.0,Home Win


## Model Train

In [22]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [23]:
feature_cols = ['win_rate_diff', 'elo_diff', 'attack_diff', 'defense_diff',
                'h2h_home_win_rate', 'rest_diff', 'is_world_cup']
X = df_features[feature_cols]
y = df_features['match_outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"[*] Training samples: {X_train.shape[0]}")
print(f"[*] Testing samples: {X_test.shape[0]}")

[*] Training samples: 21526
[*] Testing samples: 5382


In [24]:
model = RandomForestClassifier(
    n_estimators=200, max_depth=8, random_state=42, class_weight='balanced'
)
print("Training the Random Forest model")
model.fit(X_train, y_train)

Training the Random Forest model


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 T

In [25]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred))
print(f"Model accuracy: {accuracy:.2%}")

--- Detailed Classification Report ---
              precision    recall  f1-score   support

    Away Win       0.51      0.59      0.55      1530
        Draw       0.30      0.32      0.31      1259
    Home Win       0.70      0.61      0.65      2593

    accuracy                           0.54      5382
   macro avg       0.50      0.51      0.50      5382
weighted avg       0.55      0.54      0.54      5382

Model accuracy: 53.83%


In [26]:
# Hyperparameter tuning
cv_scores = cross_val_score(model, X, y, cv=5)
print("Cross-validation scores:", np.round(cv_scores, 3))
print(f"Mean CV accuracy: {cv_scores.mean():.2%} (+/- {cv_scores.std():.2%})")

Cross-validation scores: [0.469 0.534 0.518 0.56  0.572]
Mean CV accuracy: 53.06% (+/- 3.59%)


In [27]:
labels = ['Home Win', 'Draw', 'Away Win']
cm = confusion_matrix(y_test, y_pred, labels=labels)
display(pd.DataFrame(cm, index=[f"Actual: {l}" for l in labels], columns=[f"Pred: {l}" for l in labels]))

importances = model.feature_importances_
print()
print("Feature Importance")
for col, imp in sorted(zip(feature_cols, importances), key=lambda x: -x[1]):
    print(f"• {col}: {imp:.2%}")

,Pred: Home Win,Pred: Draw,Pred: Away Win
Actual: Home Win,1591,583,419
Actual: Draw,408,401,450
Actual: Away Win,269,356,905



Feature Importance
• elo_diff: 47.37%
• defense_diff: 19.40%
• win_rate_diff: 9.98%
• attack_diff: 9.42%
• h2h_home_win_rate: 7.82%
• rest_diff: 5.66%
• is_world_cup: 0.35%


## Predict the upcoming Fixtures

In [28]:
def predict_fixture(home_team, away_team, match_date, is_world_cup=1):
    """Predict an upcoming match using each team's rolling form and Elo rating as of match_date."""
    def latest_form(team):
        hist = [m for m in team_stats.get(team, []) if m[0] < match_date][-10:]
        if not hist:
            return {'win_rate': 0.33, 'goals_for': 1.0, 'goals_against': 1.0, 'days_since': 365}
        return {
            'win_rate': np.mean([m[3] for m in hist]),
            'goals_for': np.mean([m[1] for m in hist]),
            'goals_against': np.mean([m[2] for m in hist]),
            'days_since': (match_date - hist[-1][0]).days
        }

    h_form = latest_form(home_team)
    a_form = latest_form(away_team)

    h_elo = elo_ratings.get(home_team, INITIAL_ELO)
    a_elo = elo_ratings.get(away_team, INITIAL_ELO)

    pair_key = tuple(sorted([home_team, away_team]))
    h2h_hist = [m for m in h2h_stats.get(pair_key, []) if m[0] < match_date]
    if h2h_hist:
        h2h_home_results = [m[1] for m in h2h_hist if m[2] == home_team]
        h2h_rate = np.mean(h2h_home_results) if h2h_home_results else 0.5
    else:
        h2h_rate = 0.5

    features = pd.DataFrame([{
        'win_rate_diff': h_form['win_rate'] - a_form['win_rate'],
        'elo_diff': h_elo - a_elo,
        'attack_diff': h_form['goals_for'] - a_form['goals_for'],
        'defense_diff': h_form['goals_against'] - a_form['goals_against'],
        'h2h_home_win_rate': h2h_rate,
        'rest_diff': h_form['days_since'] - a_form['days_since'],
        'is_world_cup': is_world_cup
    }])[feature_cols]

    probs = model.predict_proba(features)[0]
    return dict(zip(model.classes_, probs))

In [29]:
upcoming_fixtures['date'] = pd.to_datetime(upcoming_fixtures['date_clean'])
results = []
for _, row in upcoming_fixtures.iterrows():
    probs = predict_fixture(row['home_team'], row['away_team'], row['date'])
    predicted = max(probs, key=probs.get)
    results.append({
        'date': row['date'].date(),
        'home_team': row['home_team'],
        'away_team': row['away_team'],
        'predicted_outcome': predicted,
        'home_win_prob': round(probs.get('Home Win', 0), 3),
        'draw_prob': round(probs.get('Draw', 0), 3),
        'away_win_prob': round(probs.get('Away Win', 0), 3),
    })

predictions_df = pd.DataFrame(results)
display(predictions_df)

,date,home_team,away_team,predicted_outcome,home_win_prob,draw_prob,away_win_prob
0,2026-06-19,Scotland,Morocco,Away Win,0.129,0.305,0.566
1,2026-06-19,Brazil,Haiti,Home Win,0.647,0.246,0.107
2,2026-06-19,United States,Australia,Draw,0.287,0.364,0.350
3,2026-06-19,Turkey,Paraguay,Home Win,0.445,0.347,0.207
4,2026-06-20,Germany,Ivory Coast,Home Win,0.452,0.331,0.217
5,2026-06-20,Ecuador,Curaçao,Home Win,0.663,0.236,0.101
6,2026-06-20,Netherlands,Sweden,Home Win,0.593,0.314,0.093
7,2026-06-20,Tunisia,Japan,Away Win,0.112,0.332,0.556
8,2026-06-21,Belgium,Iran,Draw,0.330,0.375,0.296
9,2026-06-21,New Zealand,Egypt,Away Win,0.138,0.290,0.572


In [30]:
def simulate_matchup(team_a, team_b, match_date, is_world_cup_match=1):
    """Simulates a knockout match, redistributing draw probability since knockouts have no draws."""
    probs = predict_fixture(team_a, team_b, match_date, is_world_cup_match)
    home_win_p = probs.get('Home Win', 0)
    away_win_p = probs.get('Away Win', 0)
    draw_p = probs.get('Draw', 0)

    if draw_p > 0:
        denom = home_win_p + away_win_p + 1e-9
        home_win_p += draw_p * (home_win_p / denom)
        away_win_p += draw_p * (away_win_p / denom)

    if home_win_p >= away_win_p:
        return team_a, home_win_p
    else:
        return team_b, away_win_p

def run_tournament(teams_list, match_date):
    """Simulates an entire knockout bracket from Quarter-Finals to the Final."""
    current_round = teams_list.copy()
    round_name = "QUARTER-FINALS"

    while len(current_round) > 1:
        print(f"\n=== SIMULATING {round_name} ===")
        print("=" * 40)
        next_round = []

        for i in range(0, len(current_round), 2):
            t1, t2 = current_round[i], current_round[i + 1]
            winner, confidence = simulate_matchup(t1, t2, match_date)
            next_round.append(winner)
            print(f"[-] {t1} vs {t2} -> Winner: {winner} ({confidence:.1%})")

        current_round = next_round
        if len(current_round) == 4:
            round_name = "SEMI-FINALS"
        elif len(current_round) == 2:
            round_name = "THE GRAND FINAL"

    print("\n" + "=" * 40)
    print(f"TOURNAMENT CHAMPION: {current_round[0]}")
    print("=" * 40)

In [31]:
quarter_final_bracket = ['Argentina', 'Croatia', 'France', 'England',
                          'Spain', 'Netherlands', 'Brazil', 'Portugal']
simulation_date = df_features['date'].max()
run_tournament(quarter_final_bracket, simulation_date)


=== SIMULATING QUARTER-FINALS ===
[-] Argentina vs Croatia -> Winner: Argentina (86.9%)
[-] France vs England -> Winner: France (58.2%)
[-] Spain vs Netherlands -> Winner: Spain (66.7%)
[-] Brazil vs Portugal -> Winner: Brazil (53.7%)

=== SIMULATING SEMI-FINALS ===
[-] Argentina vs France -> Winner: Argentina (57.4%)
[-] Spain vs Brazil -> Winner: Spain (57.0%)

=== SIMULATING THE GRAND FINAL ===
[-] Argentina vs Spain -> Winner: Argentina (56.6%)

TOURNAMENT CHAMPION: Argentina
